Recursive Generation

Install Packages

In [ ]:
%pip install -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Install scispacy model for classification (inside .venv)
# pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

In [ ]:
import accelerate
import bitsandbytes as bnb
import datasets
import numpy as np
import pandas as pd
import peft

# import spacy
# import scispacy
import torch
from tqdm.auto import tqdm
from transformers import pipeline

print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
# print("spacy:", spacy.__version__)
# print("scispacy:", scispacy.__version__)
print("torch:", torch.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)

datasets: 5.0.1
pandas: 3.0.5
numpy: 1.26.4
torch: 2.14.0
peft: 0.21.0
accelerate: 1.15.0
bitsandbytes: 0.50.2


Use Specific hardware

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using:", device)

Using: mps


In [ ]:
# PROJECT CONFIGURATION
import os

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

# G0, G1, G2, G3
NUM_GENERATIONS = 2

# LOCAL PATHS
PROJECT_DIR = "../.."
SRC_DIR = "../../src"
MODELS_DIR = "../../models"
NOTEBOOKS_DIR = "../../src/notebooks"
DATA_DIR = "../../data"
RESULTS_DIR = "../../results"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/synthetics", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/predictions", exist_ok=True)

print("Model:", MODEL_NAME)
print(f"Generations: G0 -> G{NUM_GENERATIONS}")
print("\nProject:", PROJECT_DIR)
print("Models:", MODELS_DIR)
print("Notebooks:", NOTEBOOKS_DIR)
print("Data:", DATA_DIR)
print("Results:", RESULTS_DIR)

Model: Qwen/Qwen2.5-0.5B
Generations: G0 -> G2

Project: ../..
Models: ../../models
Notebooks: ../../src/notebooks
Data: ../../data
Results: ../../results


In [ ]:
# GENERATION PATH HELPERS
def safe_model_name():
    return MODEL_NAME.replace("/", "-")


def get_model_path(generation):
    return f"{MODELS_DIR}/{safe_model_name()}-g{generation}"


def get_synthetic_file(generation):
    return f"{RESULTS_DIR}/synthetics/{safe_model_name()}-g{generation}-synthetic.csv"


def get_predictions_file(generation):
    return (
        f"{RESULTS_DIR}/predictions/{safe_model_name()}-g{generation}-predictions.csv"
    )

In [ ]:
for g in range(NUM_GENERATIONS + 1):
    print(f"G{g}")
    print("Model:", get_model_path(g))
    print("Synthetic:", get_synthetic_file(g))
    print()

G0
Model: ../../models/Qwen-Qwen2.5-0.5B-g0
Synthetic: ../../results/synthetics/Qwen-Qwen2.5-0.5B-g0-synthetic.csv

G1
Model: ../../models/Qwen-Qwen2.5-0.5B-g1
Synthetic: ../../results/synthetics/Qwen-Qwen2.5-0.5B-g1-synthetic.csv

G2
Model: ../../models/Qwen-Qwen2.5-0.5B-g2
Synthetic: ../../results/synthetics/Qwen-Qwen2.5-0.5B-g2-synthetic.csv



In [ ]:
from datasets import load_dataset

unlabeled_data = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled", split="train")
labeled_data = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

print("Unlabeled rows:", len(unlabeled_data))
print("Labeled rows:", len(labeled_data))
print(unlabeled_data)
print(unlabeled_data[0])
print(labeled_data)
print(labeled_data[0])

Unlabeled rows: 61249
Labeled rows: 1000
Dataset({
    features: ['pubid', 'question', 'context', 'long_answer'],
    num_rows: 61249
})
{'pubid': 14499029, 'question': 'Is naturopathy as effective as conventional therapy for treatment of menopausal symptoms?', 'context': {'contexts': ['Although the use of alternative medicine in the United States is increasing, no published studies have documented the effectiveness of naturopathy for treatment of menopausal symptoms compared to women receiving conventional therapy in the clinical setting.', 'To compare naturopathic therapy with conventional medical therapy for treatment of selected menopausal symptoms.', 'A retrospective cohort study, using abstracted data from medical charts.', 'One natural medicine and six conventional medical clinics at Community Health Centers of King County, Washington, from November 1, 1996, through July 31, 1998.', 'Women aged 40 years of age or more with a diagnosis of menopausal symptoms documented by a natur

In [ ]:
# Before using the labeled set as your gold evaluation set, check whether its questions/pubids overlap with the unlabeled pool. If there is overlap, we must exclude those rows from your training pool.
labeled_ids = set(labeled_data["pubid"])
unlabeled_ids = set(unlabeled_data["pubid"])

overlap = labeled_ids.intersection(unlabeled_ids)

print("Overlap:", len(overlap))
gold_eval_set = labeled_data
# Remove any gold-eval examples from the unlabeled training pool
if len(overlap) > 0:
    unlabeled_data = unlabeled_data.filter(lambda row: row["pubid"] not in overlap)
print(f"Gold eval: {len(gold_eval_set)}")

Overlap: 0
Gold eval: 1000


In [ ]:
# Create human training pool
SEED = 42

unlabeled_data_shuffled = unlabeled_data.shuffle(seed=SEED)

G0_BASE_SIZE = 2
ANCHOR_POOL_SIZE = 2
# For testing, remove the one line below
gold_eval_set = gold_eval_set[:2]

g0_human_train = unlabeled_data_shuffled.select(range(G0_BASE_SIZE))

human_anchor_pool = unlabeled_data_shuffled.select(
    range(G0_BASE_SIZE, G0_BASE_SIZE + ANCHOR_POOL_SIZE)
)

print(f"G0 train: {len(g0_human_train)}")
print(f"Anchor pool: {len(human_anchor_pool)}")
print(f"Gold eval: {len(gold_eval_set)}")

G0 train: 2
Anchor pool: 2
Gold eval: 5


In [ ]:
# VERIFY NO SPLIT OVERLAP

g0_ids = set(g0_human_train["pubid"])
anchor_ids = set(human_anchor_pool["pubid"])
gold_ids = set(gold_eval_set["pubid"])

print("G0 vs anchor:", len(g0_ids & anchor_ids))
print("G0 vs gold:", len(g0_ids & gold_ids))
print("Anchor vs gold:", len(anchor_ids & gold_ids))

G0 vs anchor: 0
G0 vs gold: 0
Anchor vs gold: 0


In [ ]:
# Prompt formatting
def format_pubmed_qa(row, answer_field="long_answer"):
    ctx = row["context"]
    if isinstance(ctx, dict) and "contexts" in ctx:
        context_str = " ".join(ctx["contexts"])
    elif isinstance(ctx, list):
        context_str = " ".join(ctx)
    else:
        context_str = str(ctx)
    return {
        "text": (
            f"### Question:\n{row['question']}\n\n"
            f"### Context:\n{context_str}\n\n"
            f"### Long Answer:\n{row[answer_field]}"
        )
    }

In [ ]:
# G0 starts with human-written answers
current_train_data = g0_human_train
current_answer_field = "long_answer"

formatted_data = current_train_data.map(
    lambda row: format_pubmed_qa(row, answer_field=current_answer_field)
)

print("Number of formatted rows:", len(formatted_data))
print("\n--- SAMPLE ---\n")
print(formatted_data[0]["text"])

Number of formatted rows: 2

--- SAMPLE ---

### Question:
Unipolar mania: a distinct entity or characteristic of manic preponderance?

### Context:
It has been reported that fewer patients with unipolar mania respond to lithium prophylaxis as do those with classical bipolar disorder. This study aimed to determine if the difference to response to lithium is related to unipolar mania or to a high preponderance of mania during the course of bipolarity. The study included bipolar-I patients (according to DSM-IV criteria) that had a ≥ 2-year history of either lithium or valproate prophylaxis as monotherapy. The response rate in the patients with unipolar mania and classical bipolar disorder were compared. Then, the response rate to lithium in all the patients with a manic episode rate<50% and>50%, and<80% and>80% during their course were compared. Finally, the above comparisons were repeated, excluding the patients with unipolar mania. The study included 121 bipolar-I patients (34 unipolar

Tokenization

In [ ]:
from transformers import AutoTokenizer

MAX_LENGTH = 1024

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def tokenize_function(row, answer_field="long_answer"):
    context_text = context_to_string(row["context"])

    prompt = (
        f"### Question:\n{row['question']}\n\n"
        f"### Context:\n{context_text}\n\n"
        f"### Long Answer:\n"
    )

    answer = str(row[answer_field])

    prompt_tokens = tokenizer(prompt, add_special_tokens=False)["input_ids"]

    answer_tokens = tokenizer(answer, add_special_tokens=False)["input_ids"]

    # Add EOS token
    answer_tokens = answer_tokens + [tokenizer.eos_token_id]

    # Reserve room for the answer
    max_prompt_length = MAX_LENGTH - len(answer_tokens)

    if max_prompt_length < 0:
        # Answer itself is too long
        answer_tokens = answer_tokens[:MAX_LENGTH]
        prompt_tokens = []
    else:
        # Truncate context/prompt instead of losing answer
        prompt_tokens = prompt_tokens[:max_prompt_length]

    input_ids = prompt_tokens + answer_tokens

    labels = [-100] * len(prompt_tokens) + answer_tokens

    attention_mask = [1] * len(input_ids)

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:
# TOKENIZATION VALIDATION
def validate_tokenization(
    tokenized_data, raw_data, answer_field, tokenizer, sample_index=0
):
    row = tokenized_data[sample_index]

    print("\n--- TOKENIZATION CHECK ---")

    print("input_ids length:", len(row["input_ids"]))
    print("labels length:", len(row["labels"]))
    print("attention_mask length:", len(row["attention_mask"]))

    # PRINT TOKEN IDS

    print("\nToken IDs:")
    print(row["input_ids"])

    # PRINT INDIVIDUAL TOKENS

    tokens = tokenizer.convert_ids_to_tokens(row["input_ids"])

    print("\nTokens:")
    print(tokens)

    # DECODE FULL INPUT

    decoded_input = tokenizer.decode(row["input_ids"], skip_special_tokens=False)

    print("\nDecoded full input:")
    print(decoded_input)

    # LOSS MASK CHECK

    ignored = row["labels"].count(-100)

    trained = sum(label != -100 for label in row["labels"])

    print("\nIgnored prompt tokens (Question + Context):", ignored)

    print("Answer tokens used for loss:", trained)

    # DECODE TRAINED ANSWER

    answer_token_ids = [label for label in row["labels"] if label != -100]

    decoded_answer = tokenizer.decode(answer_token_ids, skip_special_tokens=True)

    print("\nDecoded answer from labels:")
    print(decoded_answer)

    print("\nOriginal training answer:")
    print(raw_data[sample_index][answer_field])

In [ ]:
tokenized_data = current_train_data.map(
    lambda row: tokenize_function(row, answer_field=current_answer_field), batched=False
)

validate_tokenization(
    tokenized_data=tokenized_data,
    raw_data=current_train_data,
    answer_field=current_answer_field,
    tokenizer=tokenizer,
)


--- TOKENIZATION CHECK ---
input_ids length: 403
labels length: 403
attention_mask length: 403

Token IDs:
[14374, 15846, 510, 1806, 573, 7417, 883, 685, 25, 264, 12460, 5387, 476, 28583, 315, 69122, 855, 20328, 681, 1939, 14374, 9608, 510, 2132, 702, 1012, 4961, 429, 16745, 6835, 448, 650, 573, 7417, 883, 685, 5889, 311, 56807, 39234, 3923, 7184, 438, 653, 1846, 448, 28824, 64819, 19267, 13, 1096, 3920, 19469, 311, 8253, 421, 279, 6672, 311, 2033, 311, 56807, 374, 5435, 311, 650, 573, 7417, 883, 685, 476, 311, 264, 1550, 855, 20328, 681, 315, 883, 685, 2337, 279, 3308, 315, 64819, 487, 13, 576, 3920, 5230, 64819, 21477, 6835, 320, 73846, 311, 79167, 12, 3090, 12890, 8, 429, 1030, 264, 62147, 220, 17, 4666, 3840, 315, 2987, 56807, 476, 1044, 776, 349, 39234, 3923, 7184, 438, 1615, 41711, 13, 576, 2033, 4379, 304, 279, 6835, 448, 650, 573, 7417, 883, 685, 323, 28824, 64819, 19267, 1033, 7707, 13, 5005, 11, 279, 2033, 4379, 311, 56807, 304, 678, 279, 6835, 448, 264, 69122, 9234, 4379, 2

QLoRA setup

Load Qwen using 4-bit weights
        ↓
Use NF4 representation
        ↓
Do calculations in FP16
        ↓
Apply extra compression to quantization metadata

In [ ]:
# LOAD BASE MODEL IN 4-BIT
import torch
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

print("Base model loaded.")

Base model loaded.


In [ ]:
def create_model(generation):
    dtype = torch.float16 if torch.backends.mps.is_available() else torch.float32

    if torch.cuda.is_available():
        # Load a fresh copy of the base model
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
        )
        # Prepare quantized model for training
        model = prepare_model_for_kbit_training(model)
    else:

        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=dtype,
        )
        model.to(device)

    model.config.use_cache = False

    # Attach fresh LoRA adapters
    # G0 starts with a fresh LoRA adapter
    if generation == 0:
        model = get_peft_model(model, lora_config)
        print("Created fresh G0 adapter.")

    # G1+ loads previous generation's adapter
    else:
        previous_model_path = get_model_path(generation - 1)
        print(previous_model_path)
        model = PeftModel.from_pretrained(model, previous_model_path, is_trainable=True)
        print(f"Loaded G{generation - 1} " f"for training G{generation}")

    model.print_trainable_parameters()

    return model

Training

In [ ]:
def custom_data_collator(features):
    max_length = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_masks = []
    labels = []

    for feature in features:
        pad_length = max_length - len(feature["input_ids"])

        input_ids.append(feature["input_ids"] + [tokenizer.pad_token_id] * pad_length)

        attention_masks.append(feature["attention_mask"] + [0] * pad_length)

        labels.append(feature["labels"] + [-100] * pad_length)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

In [ ]:
from transformers import Trainer, TrainingArguments


def train_generation(generation, tokenized_data, model):
    current_model_path = get_model_path(generation)

    print(f"\n===== TRAINING G{generation} =====")
    print("Saving to:", current_model_path)

    training_args = TrainingArguments(
        output_dir=current_model_path,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="steps",
        save_steps=250,
        save_total_limit=2,
        bf16=torch.cuda.is_available(),
        gradient_checkpointing=torch.cuda.is_available(),
        torch_empty_cache_steps=50,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_data,
        data_collator=custom_data_collator,
    )

    print("Trainer created.")
    print("Training examples:", len(tokenized_data))

    trainer.train()

    # Save LoRA adapter + tokenizer
    model.save_pretrained(current_model_path)
    tokenizer.save_pretrained(current_model_path)

    print(f"G{generation} saved to:")
    print(current_model_path)

    return model

Generate synthetic answers and predicted answers

human_answer      = original reference answer

synthetic_answer  = generated answer used for recursive training

predicted_answer  = generated answer used only for evaluation

In [ ]:
def build_generation_prompt(row):
    context = row["context"]

    if isinstance(context, dict) and "contexts" in context:
        context_str = " ".join(context["contexts"])
    elif isinstance(context, list):
        context_str = " ".join(context)
    else:
        context_str = str(context)

    return (
        f"### Question:\n{row['question']}\n\n"
        f"### Context:\n{context_str}\n\n"
        f"### Long Answer:\n"
    )

In [ ]:
def generate_answer(model, tokenizer, row):
    prompt = build_generation_prompt(row)

    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    )

    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    model.eval()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Remove the prompt tokens
    generated_ids = output_ids[0][inputs["input_ids"].shape[1] :]

    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return answer

In [ ]:
from datasets import Dataset


def context_to_string(context):
    if isinstance(context, dict):
        if "contexts" in context:
            contexts = context["contexts"]

            if isinstance(contexts, (list, tuple)):
                return " ".join(str(x) for x in contexts)

            # Handles numpy arrays too
            try:
                return " ".join(str(x) for x in contexts)
            except TypeError:
                return str(contexts)

        return str(context)

    elif isinstance(context, (list, tuple)):
        return " ".join(str(x) for x in context)

    else:
        return str(context)

In [ ]:
from datasets import Dataset


def generate_dataset(model, tokenizer, source_data, output_field):
    rows = []

    for i, row in enumerate(source_data):
        generated_answer = generate_answer(model=model, tokenizer=tokenizer, row=row)
        context_text = context_to_string(row["context"])
        new_row = {
            "pubid": row["pubid"],
            "question": row["question"],
            "context": context_text,
            output_field: generated_answer,
        }

        if "long_answer" in row:
            new_row["human_answer"] = row["long_answer"]
        elif "human_answer" in row:
            new_row["human_answer"] = row["human_answer"]

        rows.append(new_row)

        print(f"Generated {i + 1}/{len(source_data)}")

    return Dataset.from_list(rows)

Loop Generations

In [ ]:
import gc

import torch
from datasets import load_dataset

for generation in tqdm(
    range(NUM_GENERATIONS + 1), colour="green", desc="Generation Loop"
):

    print(f"\n{'=' * 50}")
    print(f"GENERATION G{generation}")
    print(f"{'=' * 50}")

    # 1. LOAD TRAINING DATA
    if generation == 0:
        current_train_data = g0_human_train
        current_answer_field = "long_answer"

    else:
        current_train_data = load_dataset(
            "csv", data_files=get_synthetic_file(generation - 1), split="train"
        )
        current_answer_field = "synthetic_answer"

    print("Training rows:", len(current_train_data))
    print("Answer field:", current_answer_field)

    # 2. TOKENIZE
    tokenized_data = current_train_data.map(
        lambda row: tokenize_function(row, answer_field=current_answer_field),
        batched=False,
    )

    # 3. VALIDATE
    validate_tokenization(
        tokenized_data=tokenized_data,
        raw_data=current_train_data,
        answer_field=current_answer_field,
        tokenizer=tokenizer,
    )

    # 4. FRESH BASE MODEL + QLORA
    model = create_model(generation=generation)

    # 5. TRAIN CURRENT GENERATION
    model = train_generation(
        generation=generation, tokenized_data=tokenized_data, model=model
    )

    if isinstance(gold_eval_set, dict):
        gold_eval_set = Dataset.from_dict(gold_eval_set)

    # 6. EVALUATE ON FIXED LABELED SET
    predictions = generate_dataset(
        model=model,
        tokenizer=tokenizer,
        source_data=gold_eval_set,
        output_field="predicted_answer",
    )

    predictions.to_csv(get_predictions_file(generation), index=False)

    print("Saved predictions:", get_predictions_file(generation))

    # 7. GENERATE SYNTHETIC DATA FOR NEXT GENERATION

    synthetic_data = generate_dataset(
        model=model,
        tokenizer=tokenizer,
        source_data=current_train_data,
        output_field="synthetic_answer",
    )

    synthetic_data.to_csv(get_synthetic_file(generation), index=False)

    print("Saved synthetic data:", get_synthetic_file(generation))

    # 8. CLEAN MEMORY
    del model
    del tokenized_data

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

  0%|          | 0/3 [00:00<?, ?it/s]


GENERATION G0
Training rows: 2
Answer field: long_answer

--- TOKENIZATION CHECK ---
input_ids length: 403
labels length: 403
attention_mask length: 403

Token IDs:
[14374, 15846, 510, 1806, 573, 7417, 883, 685, 25, 264, 12460, 5387, 476, 28583, 315, 69122, 855, 20328, 681, 1939, 14374, 9608, 510, 2132, 702, 1012, 4961, 429, 16745, 6835, 448, 650, 573, 7417, 883, 685, 5889, 311, 56807, 39234, 3923, 7184, 438, 653, 1846, 448, 28824, 64819, 19267, 13, 1096, 3920, 19469, 311, 8253, 421, 279, 6672, 311, 2033, 311, 56807, 374, 5435, 311, 650, 573, 7417, 883, 685, 476, 311, 264, 1550, 855, 20328, 681, 315, 883, 685, 2337, 279, 3308, 315, 64819, 487, 13, 576, 3920, 5230, 64819, 21477, 6835, 320, 73846, 311, 79167, 12, 3090, 12890, 8, 429, 1030, 264, 62147, 220, 17, 4666, 3840, 315, 2987, 56807, 476, 1044, 776, 349, 39234, 3923, 7184, 438, 1615, 41711, 13, 576, 2033, 4379, 304, 279, 6835, 448, 650, 573, 7417, 883, 685, 323, 28824, 64819, 19267, 1033, 7707, 13, 5005, 11, 279, 2033, 4379, 311, 

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 189.93it/s]


Created fresh G0 adapter.
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497

===== TRAINING G0 =====
Saving to: ../../models/Qwen-Qwen2.5-0.5B-g0
Trainer created.
Training examples: 2


/Users/chris/Desktop/Research/recursive-llm-degradation-research/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...


--- TOKENIZATION CHECK ---
input_ids length: 404
labels length: 404
attention_mask length: 404

Token IDs:
[14374, 15846, 510, 1806, 573, 7417, 883, 685, 25, 264, 12460, 5387, 476, 28583, 315, 69122, 855, 20328, 681, 1939, 14374, 9608, 510, 2132, 702, 1012, 4961, 429, 16745, 6835, 448, 650, 573, 7417, 883, 685, 5889, 311, 56807, 39234, 3923, 7184, 438, 653, 1846, 448, 28824, 64819, 19267, 13, 1096, 3920, 19469, 311, 8253, 421, 279, 6672, 311, 2033, 311, 56807, 374, 5435, 311, 650, 573, 7417, 883, 685, 476, 311, 264, 1550, 855, 20328, 681, 315, 883, 685, 2337, 279, 3308, 315, 64819, 487, 13, 576, 3920, 5230, 64819, 21477, 6835, 320, 73846, 311, 79167, 12, 3090, 12890, 8, 429, 1030, 264, 62147, 220, 17, 4666, 3840, 315, 2987, 56807, 476, 1044, 776, 349, 39234, 3923, 7184, 438, 1615, 41711, 13, 576, 2033, 4379, 304, 279, 6835, 448, 650, 573, 7417, 883, 685, 323, 28824, 64819, 19267, 1033, 7707, 13, 5005, 11, 279, 2033, 4379, 311, 56807, 304, 678, 279, 6835, 448, 264, 69122, 9234, 4379, 2

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 412.31it/s]


../../models/Qwen-Qwen2.5-0.5B-g0
Loaded G0 for training G1
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497

===== TRAINING G1 =====
Saving to: ../../models/Qwen-Qwen2.5-0.5B-g1
Trainer created.
Training examples: 2


/Users/chris/Desktop/Research/recursive-llm-degradation-research/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
 33%|███▎      | 1/3 [00:34<01:09, 34.99s/it]


KeyboardInterrupt: 

Testing if previous model is used or not in recursion instead of new model every time

In [ ]:
import torch
from peft import PeftModel


def verify_previous_generation_loaded(model, previous_model_path):
    print("\n--- VERIFY PREVIOUS GENERATION LOAD ---")

    # Load a fresh copy of the base model
    base_check = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )

    base_check = prepare_model_for_kbit_training(base_check)

    # Load the saved previous generation adapter
    previous_model = PeftModel.from_pretrained(
        base_check, previous_model_path, is_trainable=False
    )

    loaded_params = {
        name: param.detach().cpu()
        for name, param in model.named_parameters()
        if "lora_" in name
    }

    saved_params = {
        name: param.detach().cpu()
        for name, param in previous_model.named_parameters()
        if "lora_" in name
    }

    matched = 0
    different = 0

    for name in saved_params:
        if name not in loaded_params:
            continue

        if torch.allclose(saved_params[name], loaded_params[name]):
            matched += 1
        else:
            different += 1

    print("Matching LoRA tensors:", matched)
    print("Different LoRA tensors:", different)

    if different == 0 and matched > 0:
        print(
            "SUCCESS: current model contains " "the previous generation's LoRA weights."
        )
    else:
        print("WARNING: previous-generation weights " "do not fully match.")

    del previous_model
    del base_check

In [ ]:
model = create_model(generation=generation)

if generation > 0:
    verify_previous_generation_loaded(
        model=model, previous_model_path=get_model_path(generation - 1)
    )

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 270.24it/s]


../../models/Qwen-Qwen2.5-0.5B-g1
Loaded G1 for training G2
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497

--- VERIFY PREVIOUS GENERATION LOAD ---


Loading weights: 100%|██████████| 290/290 [00:01<00:00, 199.08it/s]


Matching LoRA tensors: 336
Different LoRA tensors: 0
SUCCESS: current model contains the previous generation's LoRA weights.
